In [4]:
# Install necessary libraries
!pip install opencv-python matplotlib fer requests

import cv2
import numpy as np
import matplotlib.pyplot as plt
from fer.fer import FER
from google.colab.patches import cv2_imshow # For displaying images in Colab
import os
import requests # Added import for requests

print("Libraries installed and imported successfully.")

# --- 1. Initialize Face and Emotion Detectors ---
# Load OpenCV's Haar Cascade for face detection
# (Note: FER library often uses MTCNN for better face detection internally)
try:
    face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')
    if face_cascade.empty():
        raise IOError('Unable to load the face cascade classifier xml file.')
    print("Face cascade classifier loaded.")
except Exception as e:
    print(f"Error loading face cascade: {e}")
    print("This might affect face detection if FER's internal detector is not used.")

# Initialize the Emotion Recognizer from fer library
# This downloads a pre-trained model on its first run
emotion_detector = FER(mtcnn=True)
print("Emotion detector initialized. Model may download on first use if not already present.")

# --- 2. Function to Process an Image (Colab-friendly Preview) ---
def process_and_display_emotion(image_path):
    """
    Processes an image, detects faces and emotions, and displays the result
    using cv2_imshow for Colab compatibility.
    """
    print(f"\nProcessing image: {image_path}")
    try:
        # Load the image
        img = cv2.imread(image_path)
        if img is None:
            print(f"Error: Could not load image from {image_path}. Please check the path and file.")
            return

        # Analyze emotions in the image using FER. It handles face detection internally.
        results = emotion_detector.detect_emotions(img)

        if not results:
            print("No faces detected or no emotions found in the image.")
            cv2_imshow(img) # Show original if no faces
            return

        # Draw bounding boxes and emotion labels
        for face in results:
            x, y, w, h = face['box']
            emotions = face['emotions']

            # Get the dominant emotion
            dominant_emotion = max(emotions, key=emotions.get)
            score = emotions[dominant_emotion]

            # Draw rectangle around face
            cv2.rectangle(img, (x, y), (x + w, y + h), (0, 255, 0), 2)

            # Put text (dominant emotion and score)
            text = f"{dominant_emotion}: {score:.2f}"
            cv2.putText(img, text, (x, y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 0), 2)

            print(f"Detected face at [{x}, {y}, {w}, {h}] with emotions: {emotions}")

        # Display the result
        print("Displaying processed image:")
        cv2_imshow(img)

    except Exception as e:
        print(f"An error occurred during image processing: {e}")

# --- 3. Example Usage: Upload or use a sample image for demonstration ---
print("\n--- Image Processing Demonstration ---")
print("This section demonstrates emotion detection on a static image.")
print("You can upload your own image to Colab (e.g., 'my_face.jpg') and call:")
print("process_and_display_emotion('my_face.jpg')")

# Attempt to download a sample image if one isn't provided
sample_image_url = "https://i.stack.imgur.com/k6uYt.png" # A common test image URL
sample_image_name = "sample_face.png"

if not os.path.exists(sample_image_name):
    print(f"Downloading a sample image from {sample_image_url} using requests...")
    try:
        response = requests.get(sample_image_url, stream=True)
        response.raise_for_status() # Raise an exception for HTTP errors
        with open(sample_image_name, 'wb') as f:
            for chunk in response.iter_content(chunk_size=8192):
                f.write(chunk)
        if os.path.exists(sample_image_name):
            print(f"Downloaded '{sample_image_name}'.")
        else:
            print(f"Failed to save sample image.")
            sample_image_name = None
    except Exception as e:
        print(f"Error during sample image download: {e}")
        sample_image_name = None

# Verify the image can be read by OpenCV immediately after download
if sample_image_name and os.path.exists(sample_image_name):
    test_img_read = cv2.imread(sample_image_name)
    if test_img_read is None:
        print(f"Warning: Downloaded file '{sample_image_name}' could not be read by OpenCV. It might be corrupted or not an image.")
        sample_image_name = None # Invalidate if not readable

if sample_image_name and os.path.exists(sample_image_name):
    process_and_display_emotion(sample_image_name)
else:
    print("\nSkipping image processing demo as no sample image is available or could not be downloaded/read.")
    print("Please upload an image (e.g., 'my_face.jpg') and call: process_and_display_emotion('my_face.jpg')")


# --- 4. Live Camera Facial Emotion Detection (for Local Execution) ---
print("\n--- Live Camera Demonstration (For Local Execution) ---")
print("NOTE: The following code block is designed for local execution where OpenCV's `cv2.imshow()`")
print("can display a live camera feed. It will likely NOT work directly in Google Colab's output due to")
print("limitations in displaying real-time video streams and browser permissions.")
print("If you want to run this, please copy this section to a Python script on your local machine and execute it.")

"""
def run_live_camera_emotion_detection():
    # Attempt to open the default camera
    cap = cv2.VideoCapture(0) # 0 for default webcam
    if not cap.isOpened():
        print("Error: Could not open webcam. Make sure a camera is connected and not in use.")
        return

    print("Live camera feed started. Press 'q' to quit.")

    while True:
        ret, frame = cap.read()
        if not ret:
            print("Error: Failed to grab frame.")
            break

        # Process the frame for emotion detection
        results = emotion_detector.detect_emotions(frame)

        if results:
            for face in results:
                x, y, w, h = face['box']
                emotions = face['emotions']
                dominant_emotion = max(emotions, key=emotions.get)
                score = emotions[dominant_emotion]

                cv2.rectangle(frame, (x, y), (x + w, y + h), (0, 255, 0), 2)
                text = f"{dominant_emotion}: {score:.2f}"
                cv2.putText(frame, text, (x, y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 0), 2)

        # Display the resulting frame
        cv2.imshow('Live Facial Emotion Detection', frame)

        # Break the loop on 'q' key press
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    # Release the capture and destroy all windows
    cap.release()
    cv2.destroyAllWindows()
    print("Live camera feed stopped.")

# Uncomment the line below to run the live camera detection locally:
# run_live_camera_emotion_detection()
"""

# --- 5. Project Enhancements Discussion ---
print("\n--- Enhancing the Project ---")
print("Here are some ways to enhance this facial emotion detection system:")
print("1.  **More Robust Face Detection**: Integrate more advanced face detection models like MTCNN (which FER uses internally) or YOLO/RetinaFace for better accuracy and speed.")
print("2.  **Custom Emotion Model Training**: Train your own Convolutional Neural Network (CNN) using a larger and more diverse dataset (e.g., AffectNet, FER-2013) to improve accuracy and potentially recognize more nuanced emotions.")
print("3.  **Multi-face Tracking**: Implement object tracking (e.g., using correlation filters or deepSORT) to track emotions of multiple individuals consistently across frames.")
print("4.  **Performance Optimization**: Optimize the model for faster inference using techniques like model quantization (TensorFlow Lite, OpenVINO) or deploying on specialized hardware (GPUs, TPUs).")
print("5.  **Emotion History/Analytics**: Store and analyze emotion data over time for a person or a group, generating reports or insights.")
print("6.  **User Interface**: Develop a dedicated graphical user interface (GUI) using libraries like PyQt, Tkinter, or web frameworks (Flask, Django) for a more interactive experience.")
print("7.  **Ethical Considerations**: Address privacy concerns, bias in AI models, and ensure responsible deployment.")
print("8.  **Contextual Emotion Understanding**: Integrate other cues like body language, speech, or situational context to improve emotion recognition accuracy.")

print("\nThis provides a strong foundation. Feel free to explore specific enhancements!")


Libraries installed and imported successfully.
Face cascade classifier loaded.
Emotion detector initialized. Model may download on first use if not already present.

--- Image Processing Demonstration ---
This section demonstrates emotion detection on a static image.
You can upload your own image to Colab (e.g., 'my_face.jpg') and call:
process_and_display_emotion('my_face.jpg')

Skipping image processing demo as no sample image is available or could not be downloaded/read.
Please upload an image (e.g., 'my_face.jpg') and call: process_and_display_emotion('my_face.jpg')

--- Live Camera Demonstration (For Local Execution) ---
NOTE: The following code block is designed for local execution where OpenCV's `cv2.imshow()`
can display a live camera feed. It will likely NOT work directly in Google Colab's output due to
limitations in displaying real-time video streams and browser permissions.
If you want to run this, please copy this section to a Python script on your local machine and execut

In [20]:
from google.colab.output import eval_js
from base64 import b64decode
import numpy as np
import cv2
import json

def js_to_image(js_reply):
  if not js_reply: return None
  try:
    # Safety check for properly formatted base64 string
    parts = js_reply.split(',')
    if len(parts) < 2: return None
    image_bytes = b64decode(parts[1])
    jpg_as_np = np.frombuffer(image_bytes, dtype=np.uint8)
    return cv2.imdecode(jpg_as_np, flags=1)
  except Exception:
    return None

# --- JavaScript Bridge Setup ---
eval_js('''
  var video, canvas, div, captureCanvas, stream;

  async function startVideo() {
    stream = await navigator.mediaDevices.getUserMedia({video: true});
    div = document.createElement('div');
    div.id = 'webcam-container';
    video = document.createElement('video');
    video.style.display = 'block';
    video.srcObject = stream;
    await video.play();

    canvas = document.createElement('canvas');
    canvas.width = video.videoWidth;
    canvas.height = video.videoHeight;
    canvas.style.position = 'absolute';
    canvas.style.left = '0px';
    canvas.style.top = '0px';

    document.body.appendChild(div);
    div.appendChild(video);
    div.appendChild(canvas);
  }

  async function captureFrame() {
    if(!video || !video.srcObject || video.paused) return null;
    if(!captureCanvas) {
      captureCanvas = document.createElement('canvas');
      captureCanvas.width = video.videoWidth;
      captureCanvas.height = video.videoHeight;
    }
    const ctx = captureCanvas.getContext('2d');
    ctx.drawImage(video, 0, 0);
    return captureCanvas.toDataURL('image/jpeg', 0.7);
  }

  function stopVideo() {
    if(stream) {
      stream.getTracks().forEach(track => track.stop());
    }
    if(div) {
      div.remove();
    }
  }

  function drawRects(rectData, alertActive) {
    if(!canvas) return;
    const ctx = canvas.getContext('2d');
    ctx.clearRect(0, 0, canvas.width, canvas.height);

    if (alertActive) {
        ctx.fillStyle = "rgba(255, 0, 0, 0.5)";
        ctx.fillRect(0, 0, canvas.width, 40);
        ctx.fillStyle = "white";
        ctx.font = "bold 20px Arial";
        ctx.fillText("ALERT: PERSISTENT SADNESS", 10, 28);
    }

    ctx.strokeStyle = '#00FF00';
    ctx.lineWidth = 3;
    ctx.fillStyle = '#00FF00';
    ctx.font = 'bold 18px Arial';

    rectData.forEach(face => {
      const [x, y, w, h] = face.box;
      ctx.strokeRect(x, y, w, h);
      ctx.fillText(`${face.emotion} (${face.conf}%)`, x, y > 20 ? y - 5 : y + 20);
    });
  }
  window.drawRects = drawRects;
  window.startVideo = startVideo;
  window.captureFrame = captureFrame;
  window.stopVideo = stopVideo;
  null;
''')

eval_js('startVideo()')
print("Camera active. Press stop to end cleanly.")

try:
    frame_count = 0
    sad_counter = 0
    ALERT_THRESHOLD = 10
    rects = []

    while True:
        img_data = eval_js('window.captureFrame()')
        if img_data is None:
            break

        if frame_count % 3 == 0:
            img = js_to_image(img_data)
            if img is None: continue

            results = emotion_detector.detect_emotions(img)
            rects = []
            is_currently_sad = False

            for face in results:
                emotions = face['emotions']
                dominant = max(emotions, key=emotions.get)
                confidence = emotions[dominant]

                if confidence > 0.5:
                    rects.append({
                        'box': [int(v) for v in face['box']],
                        'emotion': dominant,
                        'conf': int(confidence * 100)
                    })
                    if dominant == 'sad': is_currently_sad = True

            sad_counter = sad_counter + 1 if is_currently_sad else 0

        alert_active = sad_counter >= ALERT_THRESHOLD
        try:
            eval_js(f'window.drawRects({json.dumps(rects)}, {json.dumps(alert_active)})')
        except:
            break
        frame_count += 1

except Exception as e:
    print(f"Error: {e}")
finally:
    eval_js('window.stopVideo()')
    print("Webcam stopped and preview removed.")

Camera active. Press stop to end cleanly.
Webcam stopped and preview removed.


KeyboardInterrupt: 